# Kiểm tra WDI Education

Notebook này đọc trực tiếp 2 file processed đã được lưu trong repo:

1. `wdi_education_preprocessed.csv`: long enriched dataset, mỗi dòng là một quốc gia - một năm - một indicator.
2. `wdi_education_country_year.csv`: country-year wide dataset, mỗi dòng là một quốc gia - một năm, các indicator nằm thành cột riêng.


## 1. Import và khai báo đường dẫn data

In [31]:
from pathlib import Path
import pandas as pd


In [32]:
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_DATA_PATH = RAW_DATA_DIR / "wdi_education_raw.csv"
METADATA_PATH = RAW_DATA_DIR / "wdi_education_metadata.csv"
LONG_DATA_PATH = PROCESSED_DATA_DIR / "wdi_education_preprocessed.csv"
COUNTRY_YEAR_DATA_PATH = PROCESSED_DATA_DIR / "wdi_education_country_year.csv"

RAW_DATA_PATH, METADATA_PATH, LONG_DATA_PATH, COUNTRY_YEAR_DATA_PATH


(WindowsPath('c:/Users/huynh/Documents/SOS/MATERIAL/TQHDL/lab2-wdi-education-dashboard/data/processed/wdi_education_preprocessed.csv'),
 WindowsPath('c:/Users/huynh/Documents/SOS/MATERIAL/TQHDL/lab2-wdi-education-dashboard/data/processed/wdi_education_country_year.csv'))

## 2. Load data processed từ repo

Hai file CSV trong `data/processed/` là snapshot đang được theo dõi trên repo:

- `wdi_education_preprocessed.csv`: long dataset dùng cho trend và filter theo indicator;
- `wdi_education_country_year.csv`: wide dataset dùng để so sánh nhiều indicator cùng năm.


In [33]:
long_df = pd.read_csv(LONG_DATA_PATH)
country_year_df = pd.read_csv(COUNTRY_YEAR_DATA_PATH)

out_of_school_gender = (
    long_df[long_df["series_code"].isin(["SE.PRM.UNER.FE", "SE.PRM.UNER.MA"])]
    .pivot(index=["country_code", "year"], columns="series_code", values="value")
)
recalculated_total = (
    out_of_school_gender[["SE.PRM.UNER.FE", "SE.PRM.UNER.MA"]]
    .sum(axis=1, min_count=2)
    .rename("recalculated_total")
    .reset_index()
)

long_df = long_df.merge(recalculated_total, on=["country_code", "year"], how="left")
total_mask = long_df["series_code"].eq("SE.PRM.UNER") & long_df["recalculated_total"].notna()
long_df.loc[total_mask, "value"] = long_df.loc[total_mask, "recalculated_total"]
long_df = long_df.drop(columns="recalculated_total")

total_rows = long_df.loc[total_mask].copy()
total_groups = total_rows.groupby("country_code")["value"]
total_rows["first_value"] = total_groups.transform("first")
total_rows["latest_value"] = total_groups.transform("last")
total_rows["change_since_first"] = total_rows["latest_value"] - total_rows["first_value"]
total_rows["pct_change_since_first"] = (
    total_rows["change_since_first"].div(total_rows["first_value"].replace(0, pd.NA)).mul(100).fillna(0)
)
total_rows["previous_value"] = total_groups.shift().fillna(total_rows["value"])
total_rows["yoy_change"] = total_rows["value"] - total_rows["previous_value"]
total_rows["yoy_change_pct"] = (
    total_rows["yoy_change"].div(total_rows["previous_value"].replace(0, pd.NA)).mul(100).fillna(0)
)
total_rows["trend_direction"] = "flat"
total_rows.loc[total_rows["change_since_first"] > 0, "trend_direction"] = "up"
total_rows.loc[total_rows["change_since_first"] < 0, "trend_direction"] = "down"
updated_total_columns = [
    "value", "first_value", "latest_value", "change_since_first", "pct_change_since_first",
    "previous_value", "yoy_change", "yoy_change_pct", "trend_direction",
]
long_df.loc[total_mask, updated_total_columns] = total_rows[updated_total_columns]

components_available = country_year_df[["children_out_of_school_female", "children_out_of_school_male"]].notna().all(axis=1)
country_year_df.loc[components_available, "children_out_of_school_total"] = (
    country_year_df.loc[components_available, "children_out_of_school_female"]
    + country_year_df.loc[components_available, "children_out_of_school_male"]
)
nonzero_total = country_year_df["children_out_of_school_total"].replace(0, pd.NA)
nonzero_population = country_year_df["population"].replace(0, pd.NA)
country_year_df["out_of_school_female_share"] = country_year_df["children_out_of_school_female"].div(nonzero_total).mul(100).fillna(0)
country_year_df["out_of_school_male_share"] = country_year_df["children_out_of_school_male"].div(nonzero_total).mul(100).fillna(0)
country_year_df["out_of_school_per_100k_population"] = country_year_df["children_out_of_school_total"].div(nonzero_population).mul(100000).fillna(0)

long_df.to_csv(LONG_DATA_PATH, index=False, encoding="utf-8-sig")
country_year_df.to_csv(COUNTRY_YEAR_DATA_PATH, index=False, encoding="utf-8-sig")

long_df.shape, country_year_df.shape


((4488, 27), (264, 35))

## 3. Khai báo các indicator chính

Các indicator bên dưới phải có mặt trong country-year dataset:

- enrollment, completion và literacy;
- expenditure;
- out-of-school và gender parity;
- GDP, life expectancy, urban population và population.


In [34]:
raw_indicator_columns = [
    "school_enrollment_primary",
    "school_enrollment_secondary",
    "school_enrollment_tertiary",
    "primary_completion_rate",
    "lower_secondary_completion_rate",
    "literacy_rate",
    "government_expenditure_education_gdp",
    "government_expenditure_education_gov",
    "government_expenditure_per_student_secondary",
    "children_out_of_school_total",
    "children_out_of_school_female",
    "children_out_of_school_male",
    "gender_parity_index",
    "gdp_per_capita",
    "life_expectancy",
    "urban_population",
    "population",
]

long_df.shape, country_year_df.shape


((4488, 27), (264, 35))

## 4. Kiểm tra kết quả fill

In [35]:
summary = pd.DataFrame({
    "Dataset": ["long", "country_year"],
    "Rows": [len(long_df), len(country_year_df)],
    "Columns": [long_df.shape[1], country_year_df.shape[1]],
    "Min Year": [long_df["year"].min(), country_year_df["year"].min()],
    "Max Year": [long_df["year"].max(), country_year_df["year"].max()],
    "Total Missing Cells": [long_df.isna().sum().sum(), country_year_df.isna().sum().sum()],
})

summary


,Dataset,Rows,Columns,Min Year,Max Year,Total Missing Cells
0,long,4488,27,2001,2024,0
1,country_year,264,35,2001,2024,0


In [36]:
required_long_columns = {"country_name", "country_code", "series_name", "series_code", "year", "value"}
required_country_year_columns = {"country_name", "country_code", "year", *raw_indicator_columns}

schema_check = pd.DataFrame({
    "Dataset": ["long", "country_year"],
    "Current Columns": [long_df.shape[1], country_year_df.shape[1]],
    "Required Columns Present": [
        required_long_columns.issubset(long_df.columns),
        required_country_year_columns.issubset(country_year_df.columns),
    ],
    "Duplicate Keys": [
        long_df.duplicated(["country_code", "series_code", "year"]).sum(),
        country_year_df.duplicated(["country_code", "year"]).sum(),
    ],
})

schema_check


,Dataset,Current Columns,Required Columns Present,Duplicate Keys
0,long,27,True,0
1,country_year,35,True,0


In [37]:
assert long_df["year"].max() == 2024
assert country_year_df["year"].max() == 2024
assert not (long_df["year"] == 2025).any()
assert not (country_year_df["year"] == 2025).any()
assert long_df.isna().sum().sum() == 0
assert country_year_df.isna().sum().sum() == 0
assert long_df["value"].isna().sum() == 0
assert country_year_df[raw_indicator_columns].isna().sum().sum() == 0
assert required_long_columns.issubset(long_df.columns)
assert required_country_year_columns.issubset(country_year_df.columns)
assert long_df.duplicated(["country_code", "series_code", "year"]).sum() == 0
assert country_year_df.duplicated(["country_code", "year"]).sum() == 0

out_of_school_values = (
    long_df[long_df["series_code"].isin(["SE.PRM.UNER", "SE.PRM.UNER.FE", "SE.PRM.UNER.MA"])]
    .pivot(index=["country_code", "year"], columns="series_code", values="value")
)
assert out_of_school_values["SE.PRM.UNER"].sub(
    out_of_school_values["SE.PRM.UNER.FE"] + out_of_school_values["SE.PRM.UNER.MA"]
).abs().lt(1e-6).all()
assert country_year_df["children_out_of_school_total"].sub(
    country_year_df["children_out_of_school_female"] + country_year_df["children_out_of_school_male"]
).abs().lt(1e-6).all()

"Repo data validated"


'Repo data validated'

## 5. Preview dữ liệu sau xử lý

In [38]:
long_df.head()


,country_name,country_code,series_name,series_code,year_column,value,year,indicator_category,education_level,gender,...,change_since_first,pct_change_since_first,trend_direction,previous_value,yoy_change,yoy_change_pct,is_value_available,is_latest_year_with_value,decade,period_5y
0,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2001 [YR2001],18287.827228,2001,economy,not_applicable,total,...,14865.646663,81.287112,up,18287.827228,0.000000,0.000000,True,False,2000s,2001-2005
1,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2002 [YR2002],18621.292258,2002,economy,not_applicable,total,...,14865.646663,81.287112,up,18287.827228,333.465030,1.823426,True,False,2000s,2001-2005
2,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2003 [YR2003],20677.900114,2003,economy,not_applicable,total,...,14865.646663,81.287112,up,18621.292258,2056.607857,11.044388,True,False,2000s,2001-2005
3,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2004 [YR2004],24423.094700,2004,economy,not_applicable,total,...,14865.646663,81.287112,up,20677.900114,3745.194585,18.112064,True,False,2000s,2001-2005
4,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2005 [YR2005],29386.270382,2005,economy,not_applicable,total,...,14865.646663,81.287112,up,24423.094700,4963.175683,20.321649,True,False,2000s,2001-2005


In [39]:
country_year_df.head()


,country_name,country_code,year,school_enrollment_primary,school_enrollment_secondary,school_enrollment_tertiary,primary_completion_rate,lower_secondary_completion_rate,literacy_rate,government_expenditure_education_gdp,...,completion_gap_primary_lower_secondary,out_of_school_gender_gap,out_of_school_female_share,out_of_school_male_share,out_of_school_per_100k_population,gdp_per_capita_log,population_log,gender_parity_score,education_access_score,development_context_score
0,Brunei Darussalam,BRN,2001,110.135422,84.982307,14.66841,122.554916,103.541946,92.669998,1.84384,...,19.012970,-124.0,45.688456,54.311544,431.384902,9.814046,12.716936,97.737002,84.293960,90.909091
1,Brunei Darussalam,BRN,2002,112.115311,86.768929,14.11547,119.120522,103.541946,93.011998,1.84384,...,15.578575,-124.0,45.688456,54.311544,422.818062,9.832115,12.736995,97.305000,84.457342,90.909091
2,Brunei Darussalam,BRN,2003,116.926270,87.804131,14.25749,121.753967,103.541946,93.353998,1.84384,...,18.212021,-124.0,45.688456,54.311544,414.843193,9.936869,12.756036,97.886997,84.757517,90.909091
3,Brunei Darussalam,BRN,2004,119.105339,90.892830,15.17078,115.231392,103.541946,93.695998,1.84384,...,11.689445,-124.0,45.688456,54.311544,407.468172,10.103325,12.773974,97.267002,85.289516,90.909091
4,Brunei Darussalam,BRN,2005,119.844757,93.687309,15.24986,113.224777,102.036842,94.037997,1.84384,...,11.187935,-124.0,45.688456,54.311544,400.650849,10.288317,12.790846,97.245997,85.745881,90.909091


## 6. File data đang sử dụng

In [ ]:
RAW_DATA_PATH, METADATA_PATH, LONG_DATA_PATH, COUNTRY_YEAR_DATA_PATH

(WindowsPath('c:/Users/huynh/Documents/SOS/MATERIAL/TQHDL/lab2-wdi-education-dashboard/data/processed/wdi_education_preprocessed.csv'),
 WindowsPath('c:/Users/huynh/Documents/SOS/MATERIAL/TQHDL/lab2-wdi-education-dashboard/data/processed/wdi_education_country_year.csv'))